# 02 — Hand-Crafted Spectral Features

Extract interpretable spectral features for the Random Forest baseline classifier. Two families of features are computed:

1. **Cross-correlation function (CCF) properties** — FWHM, bisector span, asymmetry, and peak structure of the CCF measured against a single-star template
2. **Absorption line measurements** — equivalent width, depth, FWHM, and asymmetry of key H-band lines (Mg I, Fe I, OH, etc.)

These features capture the spectroscopic signatures of binarity (line broadening, double-peaked profiles, asymmetric bisectors) in a form amenable to tree-based classifiers.

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from astropy.table import Table

# Path setup so we can import from src/
project_root = os.path.abspath(os.path.join(os.getcwd(), "../.."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from src.features import compute_ccf, measure_line_properties, extract_handcrafted_features
from src.data import get_visit_spectra
from src.preprocessing import prepare_spectrum
from config import PREPROCESS_CONFIG, FEATURE_CONFIG

%matplotlib inline
plt.rcParams.update({"figure.dpi": 120, "font.size": 11})

## 1. CCF Demonstration

The cross-correlation function (CCF) is computed by sliding a single-star template across the observed spectrum. For a single star the CCF shows a narrow, symmetric peak at the radial velocity. For a binary, the CCF is broadened, asymmetric, or double-peaked depending on the velocity separation and flux ratio.

In [ ]:
# Load labeled sample and pick one single + one binary star
labeled = Table.read(os.path.join(project_root, "data", "labeled_sample.fits"))

binaries = labeled[labeled["label"] == 1]
singles = labeled[labeled["label"] == 0]

# Pick high-SNR examples
binary_star = binaries[np.argmax(binaries["SNR"])]
single_star = singles[np.argmax(singles["SNR"])]

print(f"Single star: {single_star['APOGEE_ID'].strip()} (SNR={single_star['SNR']:.0f})")
print(f"Binary star: {binary_star['APOGEE_ID'].strip()} (SNR={binary_star['SNR']:.0f})")

# Load and preprocess spectra
spectra = {}
for name, row in [("single", single_star), ("binary", binary_star)]:
    try:
        visit = get_visit_spectra(
            row["APOGEE_ID"].strip(),
            field=row["FIELD"].strip() if "FIELD" in row.colnames else None,
        )
        flux_clean = prepare_spectrum(
            visit["flux"][0], visit["wavelength"], 
            visit["bitmask"][0] if visit["bitmask"] is not None else None,
        )
        spectra[name] = {
            "flux": flux_clean,
            "wavelength": visit["wavelength"],
            "raw_flux": visit["flux"][0],
        }
        print(f"  Loaded and preprocessed {name} star")
    except Exception as e:
        print(f"  Could not load {name} star: {e}")

# Use the high-SNR single star as the CCF template
template_flux = spectra.get("single", {}).get("flux")

# Compute CCF for both stars
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)

for ax, name in zip(axes, ["single", "binary"]):
    if name not in spectra or template_flux is None:
        ax.set_title(f"{name.title()} star — data not available")
        continue

    ccf_result = compute_ccf(spectra[name]["flux"], spectra[name]["wavelength"], template_flux)
    
    ax.plot(ccf_result["velocities"], ccf_result["ccf"], "k-", lw=1.2)
    ax.set_xlabel("Radial Velocity (km/s)")
    ax.set_title(f"{name.title()} Star CCF")
    ax.axhline(0, color="gray", ls="--", alpha=0.5)
    
    # Print CCF diagnostics
    print(f"\n{name.title()} star CCF properties:")
    for key in ["fwhm", "bisector_span", "asymmetry"]:
        if key in ccf_result:
            print(f"  {key}: {ccf_result[key]:.3f}")

axes[0].set_ylabel("CCF")
fig.suptitle("Cross-Correlation Function: Single vs Binary", fontsize=13)
fig.tight_layout()
fig.savefig(os.path.join(project_root, "figures", "ccf_comparison.png"), dpi=150, bbox_inches="tight")
plt.show()
print("Saved figures/ccf_comparison.png")

## 2. Absorption Line Properties

Measure properties of key H-band absorption lines for both stars. The `measure_line_properties()` function computes equivalent width, line depth, FWHM, and asymmetry for each specified wavelength region.

In [ ]:
# Measure absorption line properties for single vs binary
line_regions = FEATURE_CONFIG.get("line_regions", {})
print(f"Measuring {len(line_regions)} line regions from FEATURE_CONFIG\n")

rows = []
for region_name, region_def in line_regions.items():
    for star_type in ["single", "binary"]:
        if star_type not in spectra:
            continue
        props = measure_line_properties(
            spectra[star_type]["flux"],
            spectra[star_type]["wavelength"],
            region_def,
        )
        row = {"region": region_name, "star_type": star_type}
        row.update(props)
        rows.append(row)

df_lines = pd.DataFrame(rows)

# Pivot for side-by-side comparison
if not df_lines.empty:
    for metric in ["ew", "depth", "fwhm", "asymmetry"]:
        if metric in df_lines.columns:
            pivot = df_lines.pivot(index="region", columns="star_type", values=metric)
            pivot.columns = [f"{metric}_{c}" for c in pivot.columns]
            if "df_compare" not in dir():
                df_compare = pivot
            else:
                df_compare = df_compare.join(pivot)

    print("Line property comparison (single vs binary):")
    display(df_compare) if "display" in dir() else print(df_compare.to_string())
else:
    print("No spectra available for line measurement.")

## 3. Full Feature Extraction

Use `extract_handcrafted_features()` to compute the complete feature vector for each star. This combines CCF properties and absorption line measurements into a single array. For local development we run on a small subset; full extraction runs on Fornax.

In [ ]:
# Extract the full feature vector for our example stars
for star_type in ["single", "binary"]:
    if star_type not in spectra:
        print(f"Skipping {star_type} — no data loaded")
        continue

    features = extract_handcrafted_features(
        spectra[star_type]["flux"],
        spectra[star_type]["wavelength"],
        template_flux,
    )
    print(f"\n{star_type.title()} star feature vector ({len(features)} features):")
    for fname, fval in features.items():
        print(f"  {fname:30s} = {fval:.4f}")

# Extract features for a small subset of the labeled sample
N_DEMO = 50
subset = labeled[np.random.RandomState(42).choice(len(labeled), min(N_DEMO, len(labeled)), replace=False)]
print(f"\nExtracting features for {len(subset)} stars (demo subset)...")

feature_rows = []
for i, row in enumerate(subset):
    try:
        visit = get_visit_spectra(
            row["APOGEE_ID"].strip(),
            field=row["FIELD"].strip() if "FIELD" in row.colnames else None,
        )
        flux_clean = prepare_spectrum(
            visit["flux"][0], visit["wavelength"],
            visit["bitmask"][0] if visit["bitmask"] is not None else None,
        )
        feats = extract_handcrafted_features(flux_clean, visit["wavelength"], template_flux)
        feats["APOGEE_ID"] = row["APOGEE_ID"].strip()
        feats["label"] = row["label"]
        feature_rows.append(feats)
    except Exception as e:
        pass  # skip failures silently for demo

df_features = pd.DataFrame(feature_rows)
print(f"Successfully extracted features for {len(df_features)} / {len(subset)} stars")
print(f"Feature columns: {[c for c in df_features.columns if c not in ('APOGEE_ID', 'label')]}")

## 4. Feature Distributions: Binary vs Single

Visualize the distributions of the most discriminative features, split by binary/single label. Features with clear separation between the two classes will be the most useful for the Random Forest classifier.

In [ ]:
# Load pre-computed features if available, otherwise use the demo subset
features_path = os.path.join(project_root, "data", "handcrafted_features.parquet")

if os.path.exists(features_path):
    df_plot = pd.read_parquet(features_path)
    print(f"Loaded pre-computed features: {df_plot.shape}")
elif len(df_features) > 0:
    df_plot = df_features.copy()
    print(f"Using demo subset features: {df_plot.shape}")
else:
    df_plot = pd.DataFrame()
    print("No features available for plotting.")

if not df_plot.empty and "label" in df_plot.columns:
    # Identify numeric feature columns
    feature_cols = [c for c in df_plot.columns if c not in ("APOGEE_ID", "label")]

    # Select up to 8 most discriminative features by KS statistic
    from scipy.stats import ks_2samp

    ks_scores = {}
    for col in feature_cols:
        binary_vals = df_plot.loc[df_plot["label"] == 1, col].dropna()
        single_vals = df_plot.loc[df_plot["label"] == 0, col].dropna()
        if len(binary_vals) > 2 and len(single_vals) > 2:
            ks_scores[col] = ks_2samp(binary_vals, single_vals).statistic

    top_features = sorted(ks_scores, key=ks_scores.get, reverse=True)[:8]
    print(f"\nTop features by KS statistic: {top_features}")

    # Violin plots
    n_feat = len(top_features)
    ncols = 4
    nrows = (n_feat + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(16, 4 * nrows))
    axes = np.atleast_2d(axes).flatten()

    label_map = {0: "Single", 1: "Binary"}
    df_plot["class"] = df_plot["label"].map(label_map)

    for i, feat in enumerate(top_features):
        sns.violinplot(
            data=df_plot, x="class", y=feat, ax=axes[i],
            palette={"Single": "C0", "Binary": "C3"}, inner="quartile",
        )
        axes[i].set_title(f"{feat}\n(KS={ks_scores[feat]:.3f})", fontsize=10)
        axes[i].set_xlabel("")

    # Hide unused axes
    for j in range(i + 1, len(axes)):
        axes[j].set_visible(False)

    fig.suptitle("Feature Distributions: Binary vs Single", fontsize=14, y=1.01)
    fig.tight_layout()
    fig.savefig(
        os.path.join(project_root, "figures", "feature_distributions.png"),
        dpi=150, bbox_inches="tight",
    )
    plt.show()
    print("Saved figures/feature_distributions.png")

## 5. Feature Correlations

Examine the correlation structure among all hand-crafted features. Highly correlated feature pairs may indicate redundancy that could be addressed with feature selection or PCA before training the Random Forest.

In [ ]:
# Correlation matrix heatmap
if not df_plot.empty:
    feature_cols = [c for c in df_plot.columns if c not in ("APOGEE_ID", "label", "class")]
    corr = df_plot[feature_cols].corr()

    fig, ax = plt.subplots(figsize=(12, 10))
    mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
    sns.heatmap(
        corr, mask=mask, annot=True, fmt=".2f", cmap="RdBu_r",
        center=0, vmin=-1, vmax=1, square=True, ax=ax,
        cbar_kws={"shrink": 0.8, "label": "Pearson r"},
        annot_kws={"fontsize": 7},
    )
    ax.set_title("Feature Correlation Matrix", fontsize=14)
    fig.tight_layout()
    fig.savefig(
        os.path.join(project_root, "figures", "feature_correlations.png"),
        dpi=150, bbox_inches="tight",
    )
    plt.show()
    print("Saved figures/feature_correlations.png")

    # Flag highly correlated pairs
    high_corr = []
    for i in range(len(corr.columns)):
        for j in range(i + 1, len(corr.columns)):
            r = corr.iloc[i, j]
            if abs(r) > 0.85:
                high_corr.append((corr.columns[i], corr.columns[j], r))

    if high_corr:
        print(f"\nHighly correlated pairs (|r| > 0.85):")
        for f1, f2, r in sorted(high_corr, key=lambda x: -abs(x[2])):
            print(f"  {f1} <-> {f2}: r = {r:.3f}")
    else:
        print("\nNo highly correlated feature pairs found (|r| > 0.85).")
else:
    print("No features available for correlation analysis.")

In [ ]:
# Save features to parquet for downstream modeling
output_path = os.path.join(project_root, "data", "handcrafted_features.parquet")

if not df_plot.empty:
    # Drop the 'class' column if it was added for plotting
    df_save = df_plot.drop(columns=["class"], errors="ignore")
    df_save.to_parquet(output_path, index=False)
    print(f"Saved features to {output_path}")
    print(f"  Shape: {df_save.shape}")
    print(f"  Feature columns: {[c for c in df_save.columns if c not in ('APOGEE_ID', 'label')]}")
    print(f"  Labels: {df_save['label'].value_counts().to_dict()}")
else:
    print("No features to save. Run the extraction cells first.")